In [ ]:
import argparse
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader
import os

In [1]:

from trading_rest_sdk import TradingRestClient


In [11]:
import pandas as pd
client = TradingRestClient(
    api_key='ak_6b097ba7c24e003e4be3d16745144e364b752c0e03d139376455a0c1c13cf27f',
    base_url="http://61.151.241.233:8080",
    timeout=6000
)


query='''
    SELECT DISTINCT  stock_code
        FROM zz_5001 
        LIMIT 100
        '''    
result = client.execute_sql(
    sql=query
)
result = pd.DataFrame(result['data'])
result

,adj_factor,amount,amplitude,avg_ask_price,avg_bid_price,avg_volume_per_trade,bid_ask_ratio,bid_ask_spread,bid_strength,change_pct,...,trade_count,trade_date,trade_time,turnover_rate,update_date,update_time,upper_shadow,volume,vwap,zz_date
0,1.01748251,45942,0,0,0,0,0,0,0,0,...,0,2022-01-27T00:00:00+08:00,2022-01-27T14:41:00+08:00,0.0058,2022-01-27T00:00:00+08:00,2022-01-27T14:41:00+08:00,0,3900,11.78,
1,1.01748251,7072,0,0,0,0,0,0,0,0,...,0,2022-01-27T00:00:00+08:00,2022-01-27T14:42:00+08:00,0.0009,2022-01-27T00:00:00+08:00,2022-01-27T14:42:00+08:00,0,600,11.79,
2,1.01748251,27094,0,0,0,0,0,0,0,0,...,0,2022-01-27T00:00:00+08:00,2022-01-27T14:43:00+08:00,0.0034,2022-01-27T00:00:00+08:00,2022-01-27T14:43:00+08:00,0,2300,11.78,
3,1.01748251,93162,0.1695,0,0,0,0,0,0,-0.0847,...,0,2022-01-27T00:00:00+08:00,2022-01-27T14:44:00+08:00,0.0118,2022-01-27T00:00:00+08:00,2022-01-27T14:44:00+08:00,0,7900,11.79,
4,1.01748251,0,0,0,0,0,0,0,0,0,...,0,2022-01-27T00:00:00+08:00,2022-01-27T14:45:00+08:00,0,2022-01-27T00:00:00+08:00,2022-01-27T14:45:00+08:00,0,0,11.79,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,1.01748251,26048,0,0,0,0,0,0,0,0,...,0,2022-01-28T00:00:00+08:00,2022-01-28T10:45:00+08:00,0.0033,2022-01-28T00:00:00+08:00,2022-01-28T10:45:00+08:00,0,2200,11.84,
96,1.01748251,14244,0,0,0,0,0,0,0,0,...,0,2022-01-28T00:00:00+08:00,2022-01-28T10:46:00+08:00,0.0018,2022-01-28T00:00:00+08:00,2022-01-28T10:46:00+08:00,0,1200,11.87,
97,1.01748251,43956,0,0,0,0,0,0,0,0,...,0,2022-01-28T00:00:00+08:00,2022-01-28T10:47:00+08:00,0.0055,2022-01-28T00:00:00+08:00,2022-01-28T10:47:00+08:00,0,3700,11.88,
98,1.01748251,0,0,0,0,0,0,0,0,0,...,0,2022-01-28T00:00:00+08:00,2022-01-28T10:48:00+08:00,0,2022-01-28T00:00:00+08:00,2022-01-28T10:48:00+08:00,0,0,11.88,


In [ ]:

class params:
    FEATURES=['open','high','low','close','vwap','volume']
    WIN_MIN_DAYS=20
    WIN_DAY_DAYS=40
    BARS_PER_DAY=16
    PRED_HORIZON_DAYS=10
    SAMPLE_EVERY_N_DAYS=5
    TRAIN_RATIO=0.8
    LR=1e-3
    MAX_EPOCH=100
    EARLY_STOP=10
    EPS=1e-12

def read_table(path):
    if path.endswith('.parquet'):
        return pd.read_parquet(path)
    if path.endswith('.feather'):
        return pd.read_feather(path)
    return pd.read_csv(path)

def zscore_np(a,axis=0):
    mu=np.nanmean(a,axis=axis,keepdims=True)
    sd=np.nanstd(a,axis=axis,keepdims=True)
    return (a-mu)/(sd+params.EPS)

def pearson_corr_torch(x,y):
    x=x-x.mean()
    y=y-y.mean()
    vx=x.var(unbiased=False)+params.EPS
    vy=y.var(unbiased=False)+params.EPS
    return (x*y).mean()/torch.sqrt(vx*vy)

def ic_loss(pred,y):
    return -pearson_corr_torch(pred,y)

def prepare_panel(daily,min15,bars_per_day=params.BARS_PER_DAY):
    """
    daily:[date,code,open,high,low,close,vwap,volume]
    min15:[datetime,code,open,high,low,close,vwap,volume]
    输出panel:每行(code,date)一条记录，含日频特征+bars列(bars为ndarray(16,6))
    """
    daily=daily.copy()
    daily['date']=pd.to_datetime(daily['date']).dt.normalize()
    daily['code']=daily['code'].astype(str)
    daily=daily.sort_values(['code','date']).reset_index(drop=True)

    m=min15.copy()
    m['datetime']=pd.to_datetime(m['datetime'])
    m['date']=m['datetime'].dt.normalize()
    m['code']=m['code'].astype(str)
    m=m.sort_values(['code','datetime']).reset_index(drop=True)

    m['bar_id']=m.groupby(['code','date'],sort=False).cumcount()
    m=m[m['bar_id']<bars_per_day]

    min_daybars=(m.groupby(['code','date'],sort=False)
                  .apply(lambda g:g[params.FEATURES].to_numpy(dtype=np.float32))
                  .reset_index(name='bars'))

    panel=daily.merge(min_daybars,on=['code','date'],how='left')
    panel=panel[panel['bars'].notna()].reset_index(drop=True)
    return panel

class CrossSectionDatasetSimple(Dataset):
    """
    一个样本=一个交易日dt的全截面（用于IC loss）：
    X_day:(N,40,6),X_min:(N,320,6),y:(N,)
    """
    def __init__(self,panel,calendar,dates):
        self.panel=panel.copy()
        self.panel=self.panel.sort_values(['code','date']).reset_index(drop=True)
        self.panel_indexed=self.panel.set_index(['code','date']).sort_index()
        self.codes=self.panel['code'].unique().tolist()
        self.calendar=sorted(pd.to_datetime(calendar).tolist())
        self.date2pos={d:i for i,d in enumerate(self.calendar)}
        self.dates=dates

    def __len__(self):
        return len(self.dates)

    def _window_dates(self,end_date,win_days):
        p=self.date2pos.get(end_date,None)
        if p is None or p-(win_days-1)<0:
            return []
        return self.calendar[p-(win_days-1):p+1]

    def __getitem__(self,idx):
        dt=self.dates[idx]
        day_dates=self._window_dates(dt,params.WIN_DAY_DAYS)
        min_dates=self._window_dates(dt,params.WIN_MIN_DAYS)

        p=self.date2pos.get(dt,None)
        if p is None or (p+params.PRED_HORIZON_DAYS>=len(self.calendar)):
            return {'date':dt,'codes':[],'X_day':None,'X_min':None,'y':None}
        dt_fwd=self.calendar[p+params.PRED_HORIZON_DAYS]

        X_day_list=[]
        X_min_list=[]
        y_list=[]
        code_list=[]

        for code in self.codes:
            try:
                x_day=self.panel_indexed.loc[(code,day_dates),params.FEATURES].to_numpy(dtype=np.float32)
                bars_list=self.panel_indexed.loc[(code,min_dates),'bars'].tolist()
                if len(bars_list)!=params.WIN_MIN_DAYS:
                    continue
                if any((not isinstance(b,np.ndarray)) or (b.shape[0]<params.BARS_PER_DAY) for b in bars_list):
                    continue
                x_min=np.concatenate([b[:params.BARS_PER_DAY] for b in bars_list],axis=0).astype(np.float32)
                #收益率
                p0=float(self.panel_indexed.loc[(code,dt),'close'])
                p1=float(self.panel_indexed.loc[(code,dt_fwd),'close'])
                if (not np.isfinite(p0)) or (not np.isfinite(p1)) or abs(p0)<1e-12:
                    continue
                y=(p1/p0)-1.0
            except KeyError:
                continue

            X_day_list.append(x_day)
            X_min_list.append(x_min)
            y_list.append(y)
            code_list.append(code)

        if len(code_list)==0:
            return {'date':dt,'codes':[],'X_day':None,'X_min':None,'y':None}

        X_day=np.stack(X_day_list,axis=0)
        X_min=np.stack(X_min_list,axis=0)
        y=np.asarray(y_list,dtype=np.float32)

        X_day=X_day/(X_day[:,-1:,:]+params.EPS)
        X_min=X_min/(X_min.mean(axis=1,keepdims=True)+params.EPS)
        X_day=zscore_np(X_day,axis=0) #(N,40,6)
        X_min=zscore_np(X_min,axis=0) #(N,20,6)
        y=zscore_np(y,axis=0).reshape(-1) #(N)

        return {'date':dt,'codes':code_list,'X_day':torch.from_numpy(X_day),'X_min':torch.from_numpy(X_min),'y':torch.from_numpy(y)}
# 每5天提取一次样本
def make_sample_dates(calendar,start,end,every_n=params.SAMPLE_EVERY_N_DAYS):
    s=pd.to_datetime(start).normalize()
    e=pd.to_datetime(end).normalize()
    dates=[d for d in calendar if s<=d<=e]
    return dates[::every_n]

def split_train_valid(dates,train_ratio=params.TRAIN_RATIO):
    n=len(dates)
    k=int(math.floor(n*train_ratio))
    return dates[:k],dates[k:]

#15minGRU
class Model15mGRU(nn.Module):
    def __init__(self, hidden: int):
        super().__init__()
        self.enc = GRUEncoder(input_size=len(FEAT_COLS), hidden=hidden)
        self.fc = nn.Linear(hidden, 1)

    def forward(self, X_day: torch.Tensor, X_min: torch.Tensor) -> torch.Tensor:
        last, _ = self.enc(X_min)
        return self.fc(last).squeeze(-1)



class AttentionPooling(nn.Module):
   
    def __init__(self, hidden: int):
        super().__init__()
        self.W = nn.Linear(hidden, hidden)
        self.u = nn.Linear(hidden, 1, bias=False)

    def forward(self, seq: torch.Tensor):
        # seq: [N,T,H]
        u = torch.tanh(self.W(seq))        # [N,T,H]
        a = self.u(u).squeeze(-1)          # [N,T]
        w = torch.softmax(a, dim=1)        # [N,T]
        o = torch.sum(seq * w.unsqueeze(-1), dim=1)  # [N,H]
        return o




class Model15mGRUAttn(nn.Module):
    def __init__(self, hidden: int):
        super().__init__()
        self.enc = GRUEncoder(input_size=len(FEAT_COLS), hidden=hidden)
        self.attn = AttentionPooling(hidden)
        self.bn = nn.BatchNorm1d(hidden * 2)
        self.fc = nn.Linear(hidden * 2, 1)

    def forward(self, X_day: torch.Tensor, X_min: torch.Tensor) -> torch.Tensor:
        last, seq = self.enc(X_min)
        o = self.attn(seq)  # [N,H]
        cat = torch.cat([last, o], dim=-1)
        cat = self.bn(cat)
        return self.fc(cat).squeeze(-1)


class ModelHybrid(nn.Module):
  
    def __init__(self, hidden: int):
        super().__init__()
        self.enc_day = GRUEncoder(input_size=len(FEAT_COLS), hidden=hidden)
        self.enc_min = GRUEncoder(input_size=len(FEAT_COLS), hidden=hidden)
        self.fc = nn.Linear(hidden * 2, 1)

    def forward(self, X_day: torch.Tensor, X_min: torch.Tensor) -> torch.Tensor:
        d_last, _ = self.enc_day(X_day)
        m_last, _ = self.enc_min(X_min)
        cat = torch.cat([d_last, m_last], dim=-1)
        return self.fc(cat).squeeze(-1)


class ModelIncremental(nn.Module):
   
    def __init__(self, hidden: int):
        super().__init__()
        self.enc_day = GRUEncoder(input_size=len(FEAT_COLS), hidden=hidden)
        self.enc_min = GRUEncoder(input_size=len(FEAT_COLS), hidden=hidden)
        self.fc1 = nn.Linear(hidden, 1)
        self.fc2 = nn.Linear(hidden, 1)

    def forward_day(self, X_day: torch.Tensor):
        d_last, _ = self.enc_day(X_day)
        return self.fc1(d_last).squeeze(-1)

    def forward_min(self, X_min: torch.Tensor):
        m_last, _ = self.enc_min(X_min)
        return self.fc2(m_last).squeeze(-1)

    def forward(self, X_day, X_min) :
        y1 = self.forward_day(X_day)
        y2 = self.forward_min(X_min)
        return y1 + y2

def make_loader(dataset,shuffle):
    return DataLoader(dataset,batch_size=1,shuffle=shuffle,num_workers=0,collate_fn=lambda x:x[0])

@torch.no_grad()
def eval_mean_ic(model,loader,device):
    model.eval()
    ics=[]
    for b in loader:
        if b['y'] is None:
            continue
        X_day=b['X_day'].to(device)
        X_min=b['X_min'].to(device)
        y=b['y'].to(device)
        pred,_,_=model(X_min,X_day)
        ics.append(pearson_corr_torch(pred,y).item())
    return float(np.mean(ics)) if ics else float('nan')
def 
def train_simple(model,train_ds,valid_ds,device,stage='full'):
    model=model.to(device)
    opt=torch.optim.Adam([p for p in model.parameters() if p.requires_grad],lr=params.LR)

    train_loader=make_loader(train_ds,shuffle=True)
    
    valid_loader=make_loader(valid_ds,shuffle=False)

    best_ic=-1e9
    best_state=None
    bad=0

    for ep in range(1,params.MAX_EPOCH+1):
        model.train()
        for b in train_loader:
            if b['y'] is None:
                continue
            X_day=b['X_day'].to(device)
            X_min=b['X_min'].to(device)
            y=b['y'].to(device)

            opt.zero_grad(set_to_none=True)
            pred,y1,_=model(X_min,X_day)
            loss=ic_loss(y1,y) if stage=='day' else ic_loss(pred,y)
            loss.backward()
            opt.step()

        val_ic=eval_mean_ic(model,valid_loader,device)
        if not np.isfinite(val_ic):
            continue
        if val_ic>best_ic+1e-6:
            best_ic=val_ic
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            bad=0
        else:
            bad+=1
            if bad>=params.EARLY_STOP:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model

def train_incremental_two_stage(model,train_ds,valid_ds,device):
    for p in model.enc_min.parameters():
        p.requires_grad_(False)
    for p in model.fc2.parameters():
        p.requires_grad_(False)
    model=train_simple(model,train_ds,valid_ds,device=device,stage='day')

    for p in model.enc_day.parameters():
        p.requires_grad_(False)
    for p in model.fc1.parameters():
        p.requires_grad_(False)
    for p in model.enc_min.parameters():
        p.requires_grad_(True)
    for p in model.fc2.parameters():
        p.requires_grad_(True)
    model=train_simple(model,train_ds,valid_ds,device=device,stage='full')
    return model

def main():
    
    daily=read_table(args.daily_path)
    min15=read_table(args.min15_path)

    panel=prepare_panel(daily,min15,bars_per_day=params.BARS_PER_DAY)
    calendar=sorted(pd.to_datetime(daily['date']).dt.normalize().unique().tolist())

    sample_dates=make_sample_dates(calendar,args.start,args.end,every_n=params.SAMPLE_EVERY_N_DAYS)
    train_dates,valid_dates=split_train_valid(sample_dates,params.TRAIN_RATIO)

    train_ds=CrossSectionDatasetSimple(panel,calendar,train_dates)
    valid_ds=CrossSectionDatasetSimple(panel,calendar,valid_dates)

    model=ModelIncremental()
    model=train_incremental_two_stage(model,train_ds,valid_ds,device=args.device)

    val_ic=eval_mean_ic(model,make_loader(valid_ds,shuffle=False),args.device)
    print('Validation mean IC:',val_ic)

if __name__=='__main__':
    main()
